# 04 — Embeddings: Giving Tokens Meaning and Position

**Lecture goal:** understand why raw integer token IDs are a bad input for a neural network, and build the two embedding layers (token + positional) that convert a batch of IDs into the continuous vectors the model actually computes with.

## Why not just feed the integer IDs straight into the model?

Token IDs are arbitrary. If `"cat"` is token `2543` and `"dog"` is token `9981`, there is nothing meaningful about `9981` being "bigger than" `2543` — the IDs are just index labels, assigned essentially alphabetically by our vocabulary-building process in notebook 01 (and by whatever process OpenAI used for GPT-2's vocabulary in notebook 02). A neural network built on arithmetic (multiplication, addition, gradients) would badly misinterpret a raw ID as if it encoded magnitude or order.

Worse, the IDs *destroy* information we already have for free. `"cat"` and `"kitten"` are closely related words; `"dog"` and `"puppy"` likewise. Nothing about the pair of integers `2543` and `74595` reflects that. Language hands us a rich structure — some words mean nearly the same thing, others are unrelated — and integer IDs throw all of it away before the model ever sees the text.

### An analogy: why convolutional networks work so well on images

Think about recognising a cat in a photo. A CNN doesn't flatten the pixels into one long vector and hope for the best — it processes small neighbourhoods, deliberately exploiting the fact that *nearby pixels are related*: the two eyes sit close together, the whiskers are near the nose, the ears are adjacent. That spatial structure is information already present in the image, and the architecture is built to use it.

Text has its own inherent structure: **words carry meaning, and some words are closer in meaning than others.** If we don't exploit it, we're doing the equivalent of shuffling an image's pixels before training — technically learnable, but wasteful.

### What about one-hot encoding?

A natural first fix: give every token in a 50,257-word vocabulary its own 50,257-long vector of zeros with a single `1` at its ID. That does remove the bogus "9981 > 2543" ordering — but it fails the other test just as badly. Every pair of distinct one-hot vectors is *equally* far apart, so `"dog"` is exactly as unrelated to `"puppy"` as it is to `"banana"`. No similarity information is encoded at all, and the vectors are enormous and almost entirely zeros.

What we want instead: represent each token as a **vector** of many numbers (say, 256 of them) such that tokens with similar meaning or grammatical role end up with similar vectors. This is called an **embedding** (also "vector embedding", or loosely "word embedding" — though "token" is the more accurate word, since our units may be subwords or single characters). Crucially, we don't hand-design these vectors — the model *learns* them during training, the same way it learns everything else.

## How can a list of numbers possibly encode meaning?

This is the part that feels like magic until you see it concretely, so here's a hand-built example. Suppose we decide each dimension of the vector answers one question about the word — *has a tail? / is edible? / has four legs? / makes a sound? / is a pet?* — scoring high for yes and low for no. Four words, five dimensions:

| word | has a tail | is edible | four legs | makes sound | is a pet |
|---|---|---|---|---|---|
| dog | high | low | high | high | high |
| cat | high | low | high | high | high |
| apple | low | high | low | low | low |
| banana | low | high | low | low | low |

Now the *numbers themselves* carry the relationships: `dog` and `cat` agree dimension by dimension, `apple` and `banana` agree with each other, and `dog` vs. `banana` disagree almost everywhere. Similar words end up **close together in the vector space**; unrelated words end up far apart. That's exactly the information one-hot encoding and raw IDs threw away.

Two caveats before this intuition gets over-extended:

- **Nobody writes these features down.** Five hand-picked questions can't capture what 50,000 tokens mean; real embeddings use hundreds of dimensions and no individual dimension has a tidy human label. The axes are whatever training finds useful.
- **The vectors have to be *learned*.** Getting `dog`, `puppy` and `cat` near each other while pushing `banana` away, for tens of thousands of tokens at once, means training a neural network on a lot of text — which is a big part of why training these models is so expensive.

## Interlude: proof that trained embeddings really do capture meaning

Before building anything, it's worth convincing yourself that the previous section isn't wishful thinking. A classic demonstration uses **word2vec** — specifically Google's `word2vec-google-news-300`, a set of 300-dimensional vectors trained on roughly 100 billion words of news text. (Nothing below is part of the model we're building; it's a side quest for intuition. Note also that GPT-style models do *not* load pre-trained vectors like these — they train their own embeddings jointly with the rest of the network, which we'll do from notebook 10 onward.)

Three things people find striking about those vectors:

1. **Arithmetic works.** Take the vector for `king`, add `woman`, subtract `man`, and ask which word's vector is nearest the result. The answer is `queen`. Read it as: strip the masculine component, add a feminine one, keep the royalty — meaning has become something you can do algebra with.
2. **Similarity scores match intuition.** Cosine similarity is high for `woman`/`man`, `king`/`queen`, `uncle`/`aunt`, `boy`/`girl`, `nephew`/`niece` — and low for an arbitrary pair like `paper`/`water`.
3. **Distance tracks relatedness.** The length of the difference vector `‖a − b‖` is small for related words (`man`/`woman`, `nephew`/`niece`) and much larger for unrelated ones (`semiconductor`/`earthworm`). Closer in meaning really does mean closer in space.

If you'd like to reproduce this yourself, `pip install gensim` and load the vectors with `gensim.downloader` — but be warned that the download is around 1.6 GB, which is why we don't do it inside this notebook.

The takeaway: *if* the vectors are trained well, they genuinely encode semantics. Now let's see the machinery that holds them.

## The embedding layer is just a lookup table

The simplest way to think about `nn.Embedding`, PyTorch's embedding layer: it's a matrix of shape `(vocab_size, embedding_dim)` — one row per token in the vocabulary, each row being that token's vector. "Looking up" the embedding for token ID `5` means: **take row 5 of the matrix.** That's the entire operation.

The matrix starts out filled with small random numbers (we haven't trained anything yet) and only becomes meaningful once training starts updating it via gradient descent — a process we'll build starting in notebook 10.

Let's build a tiny one by hand: a pretend vocabulary of just 6 tokens, each represented by a 3-number vector.

In [11]:
import torch

torch.manual_seed(123)  # makes the "random" numbers reproducible across runs

vocab_size = 6
embedding_dim = 3

embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


`embedding_layer.weight` is the lookup table itself: a `(6, 3)` tensor — 6 rows (one per possible token ID: 0 through 5), 3 numbers per row. It's currently random because we haven't trained it on anything.

Now, "looking up" a token's embedding is just indexing into this table:

In [12]:
token_id = torch.tensor([3])
print(embedding_layer(token_id))
print()
print("Compare to row 3 of the weight matrix directly:")
print(embedding_layer.weight[3])

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

Compare to row 3 of the weight matrix directly:
tensor([-0.4015,  0.9666, -1.1481], grad_fn=<SelectBackward0>)


Identical — confirming that `nn.Embedding(token_id)` is nothing more than "fetch this row." We can also look up several token IDs at once, which is exactly what will happen with a real batch of token sequences:

In [13]:
token_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(token_ids))
print("shape:", embedding_layer(token_ids).shape)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
shape: torch.Size([4, 3])


Four token IDs in, a `(4, 3)` tensor out — four rows, one per token, each with 3 numbers. This scales up directly: for a batch of shape `(batch_size, context_length)` token IDs, the embedding layer produces `(batch_size, context_length, embedding_dim)` — one embedding vector per token, per position, per example in the batch.

### A mathematical side note

If you've seen **one-hot encoding** before: representing token ID `3` (out of 6 possible tokens) as the vector `[0, 0, 0, 1, 0, 0]`, embedding lookup is mathematically equivalent to multiplying that one-hot vector by the weight matrix — the 1 "selects" exactly one row and zeroes out the rest. `nn.Embedding` just does this selection directly by indexing, which is dramatically faster than actually performing that matrix multiplication with mostly-zero vectors.

Put another way: **an embedding layer is a `nn.Linear` layer (without bias) applied to one-hot inputs.** Feed a batch of one-hot rows `X` into `torch.nn.Linear(vocab_size, embedding_dim, bias=False)` and you get `X @ W.T` — numerically the same rows we just fetched, if the weights match. So why does PyTorch ship a separate layer at all? Efficiency. The linear version multiplies and sums 50,257 numbers per token, of which 50,256 are zero, and it needs those giant one-hot vectors materialised in memory. At GPT-2's vocabulary size that waste is enormous, so we index instead. Same math, same learnable parameters — just without doing the arithmetic we know will vanish.

## Scaling up to our real vocabulary

Only two numbers determine the whole embedding layer: the **vocabulary size** (how many rows — one per possible token) and the **embedding dimension** (how many columns — the width of each token's vector). For GPT-2's smallest model those are `50257` and `768`, giving a `(50257, 768)` **embedding weight matrix**; the largest GPT-2 widens the vectors to 1600.

Every one of those numbers is a learnable parameter. When GPT-2 was trained, nobody knew what any of them should be: the matrix was **filled with small random values**, and then *all* `50257 x 768` of them were optimized by backpropagation as part of ordinary training. It's worth being explicit about what that means — two things get learned at once. The model learns to predict the next token, and in the same gradient steps it learns the vectors it uses to represent tokens in the first place. Nothing about the semantic structure of the previous section is designed by hand; it emerges because arranging the vectors that way lowers the loss.

Our vocabulary comes from the same GPT-2 BPE tokenizer of notebook 02, so we keep `50257` rows — but we'll shrink the vectors to `256` numbers instead of 768 so the examples stay fast and readable, and revisit realistic sizes when we assemble the full model in notebook 08.

In [14]:
vocab_size = 50257
embedding_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
print("Embedding weight matrix shape:", token_embedding_layer.weight.shape)

Embedding weight matrix shape: torch.Size([50257, 256])


That's roughly 12.9 million learnable numbers (`50257 x 256`) in this one layer alone — and this is one of the *smaller* pieces of a GPT model. Let's feed it a real batch from the `DataLoader` we built in notebook 03.

In [16]:
import tiktoken
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)
        for start in range(0, len(token_ids) - context_length, stride):
            self.input_ids.append(torch.tensor(token_ids[start : start + context_length]))
            self.target_ids.append(torch.tensor(token_ids[start + 1 : start + context_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, context_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

context_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, context_length=context_length, stride=context_length, shuffle=False)

inputs, targets = next(iter(dataloader))
print("Input token IDs shape:", inputs.shape)

Input token IDs shape: torch.Size([8, 4])


In [17]:
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([8, 4, 256])


`(8, 4, 256)`: 8 examples in the batch, 4 tokens per example, 256 numbers per token. Every one of those `8 x 4 = 32` token IDs got replaced by its own 256-number vector, via a lookup in the same shared `(50257, 256)` table.

## Step 2: Positional embeddings — telling the model *where* each token is

There's a subtle but critical gap left: the embedding layer looks up a vector for `"cat"` the same way *no matter where in the sequence `"cat"` appears*. The self-attention mechanism we'll build in notebook 05 doesn't have any built-in sense of word order either — left uncorrected, "the cat chased the dog" and "the dog chased the cat" would look identical to the model!

Concretely: take "the cat sat on the mat" and "on the mat the cat sat." The token `"cat"` sits at position 1 in the first sentence and position 4 in the second, yet a plain token-embedding lookup fetches *the same row* both times — identical vectors for two words playing very different roles. Whatever extra information the ordering carries is simply thrown away.

The fix: create a *second* embedding table, this time indexed not by token identity but by **position** (0, 1, 2, 3, ... up to `context_length - 1`), and add it to the token embedding. Each position gets its own learned vector, and the model can use it to tell "this token is 1st in the sequence" apart from "this token is 3rd." If `"cat"` embeds to `x`, then in the first sentence it becomes `x + p1` and in the second `x + p4` — two different vectors for the same word, now carrying its location.

### Why the positional vector must have the same width

The positional table's second dimension is `embedding_dim` — exactly the same as the token table's. That isn't a stylistic choice: we are going to *add* the two vectors elementwise, and addition requires matching shapes. A 4-number positional vector simply cannot be added to a 256-number token vector.

> **Terminology:** "positional encoding" and "positional embedding" are used interchangeably in papers and code. Either way it means: a vector in the same high-dimensional space, standing in for a position.

In [21]:
pos_embedding_layer = torch.nn.Embedding(context_length, embedding_dim)

positions = torch.arange(context_length)
print("Position indices:", positions)

pos_embeddings = pos_embedding_layer(positions)
print("Positional embeddings shape:", pos_embeddings.shape)

pos_embeddings

Position indices: tensor([0, 1, 2, 3])
Positional embeddings shape: torch.Size([4, 256])


tensor([[-1.3632, -0.7741, -1.3922,  ..., -0.2026, -0.4602, -1.1376],
        [ 0.5553,  0.5814,  1.8575,  ...,  0.7917,  1.6529,  0.0905],
        [-0.4253, -2.9450, -0.1746,  ...,  0.8658, -0.3540, -0.7820],
        [-1.0235, -3.1168,  1.0270,  ...,  0.1064,  0.6337, -0.1116]],
       grad_fn=<EmbeddingBackward0>)

Notice the numbers above are just as random-looking as the token embedding table was — `pos_embedding_layer.weight` is a fresh `(context_length, embedding_dim)` matrix of small random values, initialized the same way `nn.Embedding` always initializes, and it has seen zero training data so far. It contains **no real "positional information" yet.**

It becomes meaningful through the exact same mechanism as the token embeddings: once we start training in notebook 10, `input_embeddings = token_embeddings + pos_embeddings` feeds forward through the whole model to produce a prediction, the loss compares that prediction to the true next token, and `loss.backward()` computes a gradient for every row of *both* embedding tables that participated. Every training batch uses every position (0 through `context_length - 1`), so every row of the positional table gets updated on every step — gradually shaping each position's vector into whatever representation actually helps the model predict the next token better. Position 0 "learns" to look different from position 5 only because doing so lowers the loss; nobody hand-designs what these vectors should mean.

### Note: only `context_length` position vectors are ever needed

It's worth pausing on why this table has just 4 rows while the token table had 50,257. The token table needs one row per *possible token*, and any of the 50,257 could show up. The positional table needs one row per *slot in the input window* — and the model never sees more than `context_length` tokens at once. Whichever example in the batch we look at, its first token is at position 0, its second at position 1, and so on. So the same four vectors are reused for every sequence, in every batch, forever.

## Two families of positional encoding

What we just built is an **absolute** positional embedding: every position in the window gets its own dedicated vector, added to whatever token happens to land there. The alternative is **relative** positional encoding, where what gets encoded is the *distance between* two tokens — "three tokens apart" — rather than "this one is 5th."

The trade-off:

- **Absolute** is simple and works well when the fixed order of tokens matters, as in sequence generation. Its weakness is length: we learn exactly `context_length` position vectors, so a longer sequence at inference time has positions the model has literally never seen.
- **Relative** generalizes to sequence lengths not seen during training, since only offsets matter, and tends to suit long documents where the same phrase can recur in many places.

Either beats no positional information at all. In practice absolute is the more common choice, and it's what the GPT family uses — so it's what we'll implement.

### Learned vs. fixed formula

There's a second axis, easy to conflate with the first. The original Transformer paper (*Attention Is All You Need*) also used absolute positions, but computed them from a **fixed sinusoidal formula** — sines and cosines of the position index, nothing learned.

GPT drops the formula entirely. The positional table is just parameters, randomly initialized as we saw above, and **optimized during training exactly like the token embeddings.** Both tables are things the model figures out for itself; neither is designed by hand. That's the approach in the cell above, and it's why the numbers printed there are meaningless for now.

## Step 3: Combining token and positional embeddings

Now we add the two together, elementwise. Notice the shapes don't match exactly: token embeddings are `(8, 4, 256)` and positional embeddings are `(4, 256)`. PyTorch handles this automatically via **broadcasting**: when shapes don't match but are "compatible" (here, the trailing dimensions `(4, 256)` line up exactly, and the missing leading dimension is implicitly treated as size 1), PyTorch conceptually repeats the smaller tensor across the mismatched dimension — the *same* positional-embedding table gets added to every example in the batch, since position 0 means the same thing regardless of which example it's in.

In [8]:
input_embeddings = token_embeddings + pos_embeddings
print("Combined input embeddings shape:", input_embeddings.shape)

Combined input embeddings shape: torch.Size([8, 4, 256])


`(8, 4, 256)` — same shape as the token embeddings alone, because broadcasting added the `(4, 256)` positional table into every one of the 8 examples. **This is the actual tensor that gets fed into the first transformer block** — it now encodes both *what* each token is and *where* it sits in the sequence.

## Recap

- Raw integer token IDs carry no meaningful numeric relationship — and one-hot vectors are no better, since every pair is equally distant. Both throw away the fact that some words mean nearly the same thing, the way flattening pixels would throw away an image's spatial structure.
- Representing a token as a vector *can* encode meaning: similar tokens land close together in the space, which is why trained vectors support things like `king - man + woman ≈ queen`.
- `nn.Embedding` is a learnable lookup table of shape `(vocab_size, embedding_dim)`; looking up a token ID means fetching that row. It's mathematically a bias-free linear layer over one-hot inputs, but indexing skips the multiply-by-zero work.
- The table is initialized randomly and its every entry is optimized by backpropagation during LLM training — the model learns its own representations alongside learning to predict the next token.
- Token embeddings alone don't encode word order: `"cat"` gets the same vector wherever it appears. A second lookup table, indexed by **position** instead of token identity, is added elementwise (via broadcasting) to inject that information — which is why it must share the same `embedding_dim`.
- This table needs only `context_length` rows, because that's the largest input window the model ever sees, and the same rows are reused across every example.
- Positional schemes split into **absolute** (a vector per slot; GPT's choice) and **relative** (encoding distance between tokens, generalizing better to unseen lengths), and separately into **fixed sinusoidal** formulas (original Transformer) versus **learned** tables (GPT).
- The result — `input_embeddings`, shape `(batch_size, context_length, embedding_dim)` — is the true starting point for everything that follows.

### What's next

We finally have vectors the model can compute with. In notebook 05, we'll build the mechanism that made transformers famous: **self-attention** — letting each token's vector be updated based on *which other tokens in the sequence it should pay attention to.*